In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from backend.agents.extractor import Extractor

In [15]:
from pydantic import BaseModel, Field
from backend.agents.extractor import Extractor
from typing import Optional
from backend.llms.ollama import OllamaLLM, OpenAIOutputMessage, LlamaOutputMessage
from backend.llms.bedrock import BedrockNova
import logging
from backend.utils import setup_logger
from backend.prompt_hub import PromptHub
setup_logger(logging.DEBUG)

openai_llm = OllamaLLM(model_name="gpt-oss:20b", OutputMessage=OpenAIOutputMessage)
llama_llm = OllamaLLM(model_name="llama3.2-vision:11b", OutputMessage=LlamaOutputMessage)
nova_llm = BedrockNova(model_id="us.amazon.nova-micro-v1:0")

✅ Development logging enabled (DEBUG level) - Console


In [16]:
from backend.websocket_tasks.extraction_data_model import CustomerInfo, CustomerInterest

In [17]:
demographic_extractor = Extractor(
    agent_name="demographic_extractor",
    # llm=openai_llm,
    # llm=llama_llm,
    llm=nova_llm,
    system_prompt=PromptHub().extract_customer_information,
    DataModel=CustomerInfo,
)

In [19]:
content = """\
TEXT: สวัสดีครับผมอนันต์จากบริษัทประกันภัย เรียนสายคุยสมชาย ไม่ทราบว่าสะดวกคุยไหมครับ? สะดวกครับ ผมอยากรบกวนสอบถามว่าตอนนี้คุณสมชายอยาุเท่าไหร่แล้วครับ ตอนนี้ผมอายุสามสิบห้าปีแล้วครับ
""".strip()
response = demographic_extractor.run([{"role": "user", "content": content}])
response

2025-11-23 21:36:03,410 - demographic_extractor - run:35 - INFO - start extracting...
2025-11-23 21:36:05,739 - demographic_extractor - run:40 - INFO - extraction successful on attempt 1
2025-11-23 21:36:05,740 - demographic_extractor - run:60 - INFO - end extracting with input_tokens=598 and output_tokens=36


CustomerInfo(age=35, income_per_month=None)

In [21]:
def token_calculation(input_tokens, output_tokens, input_price, output_price):
    return (input_tokens * input_price + output_tokens * output_price)

token_calculation(
    input_tokens=demographic_extractor.input_tokens,
    output_tokens=demographic_extractor.output_tokens,
    input_price=0.0000011550,
    output_price=0.000004620,
)

0.00085701

In [7]:
demographic_extractor.input_tokens, demographic_extractor.output_tokens

(0, 0)